# Koddan Kasa · Levha 01 · Bu masa, iki küp ve yerçekimi: 15 satır yazı

Simülasyondaki dünya bir metin dosyası. Fizik motoru (MuJoCo) bu dosyayı okur ve adım adım çözer.

Videodaki deney: aynı dosyada bir küpün kütlesini 10 kat yapıp iki küpü aynı anda bırakıyoruz. Sonra dosyaya havayı ekliyoruz.

## 1 · Kurulum

In [ ]:
!pip -q install mujoco==3.17.0

In [ ]:
import json, mujoco, numpy as np
print('mujoco', mujoco.__version__, '· numpy', np.__version__)
# deney deterministik: rastgelelik yok, seed gerekmiyor

## 2 · Dünya: 15 satır

Zemin, masa (70×50 cm, üstü 40 cm), iki 20 cm küp (ikisi de 100 g), yerçekimi. `timestep` yazılmadı: MuJoCo varsayılanı 0,002 sn = saniyede 500 adım. `density` de yazılmadı: varsayılanı 0, yani hava yok.

In [ ]:
XML = """<mujoco>
 <option gravity="0 0 -9.81"/>
 <worldbody>
  <geom type="plane" size="2 2 .1"/>
  <geom name="masa" type="box" size=".35 .25 .4"/>
  <body name="hafif" pos="-.13 0 .9">
   <freejoint/>
   <geom type="box" size=".1 .1 .1" mass=".1"/>
  </body>
  <body name="ağır" pos=".13 0 .9">
   <freejoint/>
   <geom type="box" size=".1 .1 .1" mass=".1"/>
  </body>
 </worldbody>
</mujoco>"""
LINES = XML.split("\n")
print(len(LINES), "satır")

## 3 · Bırak ve ölç

`make_xml` videodaki canlı kod değişikliğinin aynısı: 8. ve 12. satırdaki `mass`, 2. satıra `density`.

In [ ]:
TABLE_TOP, HALF = 0.40, 0.10

def fmt_mass(kg):
    s = f"{kg:g}"
    return s[1:] if s.startswith("0.") else s

def make_xml(light_kg=0.1, heavy_kg=1.0, air=False):
    lines = list(LINES)
    lines[7] = lines[7].replace('mass=".1"', f'mass="{fmt_mass(light_kg)}"')
    lines[11] = lines[11].replace('mass=".1"', f'mass="{fmt_mass(heavy_kg)}"')
    if air:
        lines[1] = ' <option gravity="0 0 -9.81" density="1.2"/>'   # havanın yoğunluğu, kg/m³
    return "\n".join(lines)

def drop(xml, sim_sec=1.0):
    m = mujoco.MjModel.from_xml_string(xml)
    d = mujoco.MjData(m)
    dt = m.opt.timestep
    adr = {n: m.jnt_qposadr[m.body_jntadr[m.body(n).id]] for n in ("hafif", "ağır")}
    zs = {n: [d.qpos[a + 2]] for n, a in adr.items()}
    while d.time < sim_sec - 1e-9:
        mujoco.mj_step(m, d)
        for n, a in adr.items():
            zs[n].append(d.qpos[a + 2])
    out = {}
    for n, z in zs.items():
        bottom = np.array(z) - HALF
        i = int(np.argmax(bottom <= TABLE_TOP))          # alt yüzün masaya indiği adım
        t_cross = (i - 1 + (bottom[i - 1] - TABLE_TOP) / (bottom[i - 1] - bottom[i])) * dt
        out[n] = dict(step=i, cross_ms=t_cross * 1000, z=np.array(z))
    k = out["ağır"]["step"]
    out["gap_cm"] = (out["hafif"]["z"][k] - out["ağır"]["z"][k]) * 100
    out["diff_ms"] = out["hafif"]["cross_ms"] - out["ağır"]["cross_ms"]
    return out

for name, xml in [("eşit (100 g, 100 g)", make_xml(0.1, 0.1)), ("vakum (100 g, 1 kg)", make_xml(0.1, 1.0)), ("hava (100 g, 1 kg)", make_xml(0.1, 1.0, air=True))]:
    r = drop(xml)
    print(f"{name:20s} hafif adım {r['hafif']['step']} · ağır adım {r['ağır']['step']} · fark {r['diff_ms']:.2f} ms · ağır indiğinde hafif {r['gap_cm']:.2f} cm yukarıda")

Beklenen (videodaki sayılar): vakumda ikisi de **143. adımda** iner, fark 0. Havada ağır 143'te, hafif 145'te; ağır indiğinde hafif **1,1 cm** yukarıda (ara değerle 4,08 ms).

## 4 · Tarama: hafif küpün kütlesi

Ağır küp hep 1 kg. Videonun sorusu: 20 gramlık kâğıt küp kaç ms geç kalır? Önce tahmin et, sonra koştur.

In [ ]:
for kg in [0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1.0]:
    v = drop(make_xml(kg, 1.0, air=False))["diff_ms"]
    a = drop(make_xml(kg, 1.0, air=True))["diff_ms"]
    print(f"hafif {kg*1000:6.0f} g · vakum {v:6.2f} ms · hava {a:6.2f} ms")

## 5 · Sonuç dosyası

Videodaki her sayı bu dosyadan (`results/01/result.json`).

In [ ]:
import os
r_v, r_a = drop(make_xml(0.1, 1.0)), drop(make_xml(0.1, 1.0, air=True))
res = dict(mujoco=mujoco.__version__, line_count=len(LINES),
           vakum=dict(light_step=r_v["hafif"]["step"], heavy_step=r_v["ağır"]["step"], diff_ms=round(r_v["diff_ms"], 3)),
           hava=dict(light_step=r_a["hafif"]["step"], heavy_step=r_a["ağır"]["step"], diff_ms=round(r_a["diff_ms"], 3), gap_cm=round(r_a["gap_cm"], 3)),
           paper_20g_ms=round(drop(make_xml(0.02, 1.0, air=True))["diff_ms"], 2))
os.makedirs("results/01", exist_ok=True)
json.dump(res, open("results/01/result.json", "w"), ensure_ascii=False, indent=1)
res

## Dene
- Küpleri 2 m yükseğe koy (`pos="... 2.4"`): havada fark ne kadar büyüyor?
- `density="1000"` (su): küpler ne yapıyor?
- `gravity="0 0 -1.62"` (Ay): vakumda iki küp yine aynı adımda mı iniyor?
- Sınır: MuJoCo'nun hava modeli basit bir yaklaşım (katı, dönmeyen küp). Gerçek bir kâğıt küp yalpalar.